In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

!pip install pyspark

# **loading the data set**


In [ ]:
from pyspark.sql import SparkSession


In [ ]:
spark = SparkSession.builder.appName("SparkByExamples.com").getOrCreate()

# lode files from the drive


In [ ]:
arti_clean = spark.read.parquet("/content/drive/MyDrive/h&m_project/articles_cleaned.parquet")

In [ ]:
cust_clean = spark.read.parquet("/content/drive/MyDrive/h&m_project/customers_cleaned.parquet")
#

In [ ]:
tran_interaction = spark.read.parquet("/content/drive/MyDrive/h&m_project/tran_interaction.parquet")

In [ ]:
tran_filter = spark.read.parquet("/content/drive/MyDrive/h&m_project/transactions_cleaned.parquet")

In [ ]:
user_profile = spark.read.parquet("/content/drive/MyDrive/h&m_project/user_profiles.parquet")

In [ ]:
print('DATA LOAD SUCCESFULLY')

In [ ]:
tran_interaction.columns

In [ ]:
user_profile.columns

In [ ]:
cust_clean.columns

# **JOINING THE TABLES**

In [ ]:
master_df = tran_interaction.join(user_profile, on="customer_id", how="inner")

In [ ]:
master_df.columns

In [ ]:
master_df = master_df.join(cust_clean, on="customer_id", how="inner")

In [ ]:
master_df = master_df.join(arti_clean, on="article_id", how="inner")

In [ ]:
master_df.columns

In [ ]:
master_df.show()

# Re-Indexing the IDs

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
user_map = master_df.select("customer_id").distinct().orderBy("customer_id") \
    .withColumn("user_idx", F.row_number().over(Window.orderBy("customer_id")) - 1)

In [ ]:
item_map = master_df.select("article_id").distinct().orderBy("article_id") \
    .withColumn("item_idx", F.row_number().over(Window.orderBy("article_id")) - 1)

In [ ]:
master_df = master_df.join(user_map, on="customer_id", how="inner") \
                     .join(item_map, on="article_id", how="inner")

In [ ]:
print("Unique Users:", master_df.select("user_idx").distinct().count())
print("Unique Items:", master_df.select("item_idx").distinct().count())

# **#creating the files for TFR**

In [ ]:
from pyspark.ml.feature import StringIndexer

In [ ]:
interaction = master_df.select("user_idx", "item_idx").distinct()

In [ ]:
user_features = master_df.select("user_idx", "lifetime_value", "age_group", "avg_spending",'avg_shopping_interval').drop_duplicates(['user_idx'])

In [ ]:
item_features = master_df.select("item_idx",'product_group_name','popularity_score').drop_duplicates(['item_idx'])

## **# saving the files for Two-Tower model**

In [ ]:
interaction.write.mode("overwrite").parquet("/content/drive/MyDrive/h&m_project/interaction.parquet")
print("intrections.parquet saved! Ready for two-tower Model.")

In [ ]:
user_features.write.mode("overwrite").parquet("/content/drive/MyDrive/h&m_project/user_features.parquet")
print("user_features.parquet saved! Ready for two-tower Model.")

In [ ]:
item_features.write.mode("overwrite").parquet("/content/drive/MyDrive/h&m_project/item_features.parquet")
print("item_features.parquet saved! Ready for two-tower Model.")